Cell 1 — Imports & Config

In [0]:
import requests
import json
import time
from datetime import datetime

SECRET_SCOPE = "flightliability-kv-scope"
API_KEY_SECRET_NAME = "delay-liability-apikey"  

STORAGE_ACCOUNT = "stflightliabilitydev01"
CONTAINER = "raw"
BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/airlabs/delays"

API_URL = "https://airlabs.co/api/v9/delays"
MIN_DELAY =  "60"
DELAY_TYPES = ["arrivals"]
LIMIT = 50

MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = [5, 15, 30]
REQUEST_TIMEOUT_SECONDS = 30

Cell 2 — Logger

In [0]:
def log(level, message, **kwargs):
    entry = {
        "timestamp": datetime.utcnow().isoformat(),
        "level": level,
        "message": message,
        **kwargs
    }
    print(json.dumps(entry))

log("INFO", "Ingestion run started")

{"timestamp": "2026-09-01T02:32:30.305324", "level": "INFO", "message": "Ingestion run started"}


/home/spark-fdf1a243-681a-4bdd-a404-b9/.ipykernel/104/command-6848597781365436-342237913:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


Cell 3 — Retrieve API key from Key Vault

In [0]:
try:
    api_key = dbutils.secrets.get(scope=SECRET_SCOPE, key=API_KEY_SECRET_NAME)
except Exception as e:
    log("ERROR", "Failed to retrieve API key from Key Vault", error=str(e))
    raise

Cell 4 — Call the API, with retry + backoff

In [0]:
def fetch_and_write_delays(delay_type):
    """Handles one full API call + write cycle for a single delay type (departures or arrivals)."""

    params = {
        "api_key": api_key,
        "delay": MIN_DELAY,
        "type": delay_type,
        "limit": LIMIT
    }

    response = None
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            log("INFO", f"API call attempt {attempt}/{MAX_RETRIES}", delay_type=delay_type)
            response = requests.get(API_URL, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
            if response.status_code == 200:
                break
            else:
                last_error = f"HTTP {response.status_code}: {response.text[:500]}"
                log("WARNING", "Non-200 response", status_code=response.status_code, attempt=attempt, delay_type=delay_type)
        except requests.exceptions.Timeout:
            last_error = "Request timed out"
            log("WARNING", "Request timeout", attempt=attempt, delay_type=delay_type)
        except requests.exceptions.ConnectionError as e:
            last_error = f"Connection error: {e}"
            log("WARNING", "Connection error", attempt=attempt, error=str(e), delay_type=delay_type)
        except requests.exceptions.RequestException as e:
            last_error = f"Request exception: {e}"
            log("WARNING", "Request exception", attempt=attempt, error=str(e), delay_type=delay_type)

        if attempt < MAX_RETRIES:
            sleep_time = RETRY_BACKOFF_SECONDS[attempt - 1]
            log("INFO", f"Retrying in {sleep_time}s", delay_type=delay_type)
            time.sleep(sleep_time)

    if response is None or response.status_code != 200:
        log("ERROR", "All retry attempts failed", delay_type=delay_type, last_error=last_error)
        raise RuntimeError(f"AirLabs API call failed for {delay_type} after {MAX_RETRIES} attempts: {last_error}")

    try:
        data = response.json()
    except json.JSONDecodeError as e:
        log("ERROR", "Failed to parse API response as JSON", error=str(e), delay_type=delay_type)
        raise

    if isinstance(data, dict) and "error" in data:
        log("ERROR", "AirLabs API returned an error payload", api_error=data["error"], delay_type=delay_type)
        raise RuntimeError(f"AirLabs API error ({delay_type}): {data['error']}")

    records = data.get("response", data) if isinstance(data, dict) else data
    record_count = len(records)
    log("INFO", "API call succeeded", records_fetched=record_count, delay_type=delay_type)

    if record_count == 0:
        log("WARNING", "API returned zero records — nothing to write", delay_type=delay_type)
        return

    now = datetime.utcnow()
    ingest_date = now.strftime("%Y-%m-%d")
    ingest_ts = now.strftime("%Y%m%d_%H%M%S")
    file_path = f"{BASE_PATH}/ingest_date={ingest_date}/delays_{delay_type}_{ingest_ts}.json"

    # --- Idempotency check step (kept as a clearly separated step, still inside the function) ---
    try:
        dbutils.fs.ls(file_path)
        log("WARNING", "File already exists — skipping write", path=file_path, delay_type=delay_type)
        return
    except Exception:
        pass

    # --- Write step ---
    try:
        dbutils.fs.put(file_path, json.dumps(data), overwrite=False)
        log("INFO", "Write succeeded", path=file_path, record_count=record_count, delay_type=delay_type)
    except Exception as e:
        log("ERROR", "Failed to write file to storage", path=file_path, error=str(e), delay_type=delay_type)
        raise

Cell 5 — Fetch Function Call

In [0]:
for delay_type in DELAY_TYPES:
    fetch_and_write_delays(delay_type)

log("INFO", f"Ingestion run completed — processed {delay_type}")

/home/spark-fdf1a243-681a-4bdd-a404-b9/.ipykernel/104/command-6848597781365436-342237913:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


{"timestamp": "2026-09-01T02:32:32.828599", "level": "INFO", "message": "API call attempt 1/3", "delay_type": "arrivals"}
{"timestamp": "2026-09-01T02:32:33.444057", "level": "INFO", "message": "API call succeeded", "records_fetched": 100, "delay_type": "arrivals"}


/home/spark-fdf1a243-681a-4bdd-a404-b9/.ipykernel/104/command-6848597781365438-825474711:60: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


Wrote 104797 bytes.
{"timestamp": "2026-09-01T02:32:52.021725", "level": "INFO", "message": "Write succeeded", "path": "abfss://raw@stflightliabilitydev01.dfs.core.windows.net/airlabs/delays/ingest_date=2026-09-01/delays_arrivals_20260901_023233.json", "record_count": 100, "delay_type": "arrivals"}
{"timestamp": "2026-09-01T02:32:52.021918", "level": "INFO", "message": "Ingestion run completed \u2014 processed arrivals"}


/home/spark-fdf1a243-681a-4bdd-a404-b9/.ipykernel/104/command-6848597781365436-342237913:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
